# 🐾 Animal Sound Generator — v16

**More data + latent diffusion.** 1000+ samples per class.

| Step | Time |
|------|------|
| Mount Drive + unzip data | ~2 min |
| Train Decoder | ~30 min |
| Train Latent Diffusion | ~30 min |
| Generate & Download | ~2 min |

### First time: build data zip
Run `colab/build_data.ipynb` once to download ESC-50 + UrbanSound8K + Xeno-Canto.
Saves `animal1000.zip` to Drive — then this notebook just unzips it.

In [ ]:
# @title 1. Setup + Mount Drive + Unzip Data
!git clone https://github.com/weseegod/animal_sound_generator.git /content/animal_sound_generator
%cd /content/animal_sound_generator
!git pull

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib tqdm soundfile
!mkdir -p models

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

# Unzip training data from Drive
ZIP = "/content/drive/MyDrive/animal_sound_generator/data/animal1000.zip"
import os
if os.path.exists(ZIP):
    !unzip -qo "{ZIP}" -d data/
    print("✅ Data unzipped from Drive")
    !ls data/animal1000/
else:
    print("⚠️ animal1000.zip not found in Drive")
    print("   Run colab/build_data.ipynb first to create it")

# Restore encoder checkpoint from Drive (568MB)
DRIVE_MODELS = "/content/drive/MyDrive/animal_sound_generator/models"
if os.path.isdir(DRIVE_MODELS):
    !cp {DRIVE_MODELS}/best_autoencoder_train.pth models/ 2>/dev/null
    if os.path.exists('models/best_autoencoder_train.pth'):
        print('✅ Encoder checkpoint restored from Drive')

In [ ]:
# @title 2. Phase 1: Train Decoder (~30 min)
!python src/latent_diff/train_decoder.py

In [ ]:
# @title 3. Phase 2: Train Latent Diffusion (~30 min)
!python src/latent_diff/train_diff.py

In [ ]:
# @title 4. Generate & Download
!python src/latent_diff/generate.py

import zipfile, os
with zipfile.ZipFile('v16_animals.zip', 'w') as z:
    for f in os.listdir('outputs/generated'):
        if f.endswith('.wav'): z.write(f'outputs/generated/{f}', f)
from google.colab import files
files.download('v16_animals.zip')

# Save checkpoints to Drive
!cp models/latent_decoder_best.pth models/latent_diffusion_best.pth {DRIVE_MODELS}/ 2>/dev/null
print('✅ Checkpoints saved to Drive')